# Lab 06 Prelab Walkthrough — Uncertainty Propagation and the Power Rule

Work through this notebook **before** your Lab 06 session (Week 1), running
every cell and reading every explanation. It teaches this lab's *new* skills
on a toy problem — no hardware needed. The **CHECKPOINT** boxes ask for
specific values you will report in the **Prelab quiz on Canvas**.

Skills covered:

1. Propagating measurement uncertainty into a computed result
2. The **power rule** — the fast path for products, quotients, and powers
3. Building an **uncertainty budget** from a list of tuples and one loop
4. Bar charts with `ax.bar`
5. Pairing neighboring readings with `v[:-1]` and `v[1:]` (the bracketing trick)

Run the setup cell first:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 10

## 1. The toy problem: density of a cylinder

You measure an aluminum cylinder to determine its density
$\rho = m / V = 4m / (\pi d^2 h)$:

| quantity | value | instrument | uncertainty (half a resolution step) |
|---|---|---|---|
| mass $m$ | 68.20 g | balance (0.1 g) | ±0.05 g |
| diameter $d$ | 25.40 mm | caliper (0.02 mm) | ±0.01 mm |
| height $h$ | 49.85 mm | ruler (0.5 mm) | ±0.25 mm |

First, the answer itself:

In [ ]:
m = 68.20e-3      # kg
d = 25.40e-3      # m
h = 49.85e-3      # m

rho = 4 * m / (np.pi * d**2 * h)
print(f"rho = {rho:.1f} kg/m^3   (aluminum: ~2700)")

> **CHECKPOINT 1:** Report the computed density to 1 decimal (kg/m³).

## 2. How wrong could it be? The power rule

Each measurement carries uncertainty, and each contributes to uncertainty
in $\rho$. The general recipe (RSS propagation) needs partial derivatives —
but our formula is a pure **product of powers**:

$$\rho = \frac{4}{\pi} \; m^{+1} \, d^{-2} \, h^{-1}$$

and for that shape the propagation collapses to the **power rule**:

$$\frac{u_\rho}{\rho} = \sqrt{ \left(1\cdot\frac{u_m}{m}\right)^2 +
\left(2\cdot\frac{u_d}{d}\right)^2 + \left(1\cdot\frac{u_h}{h}\right)^2 }$$

Each variable's *relative* (fractional) uncertainty, times the magnitude of
its exponent, combined root-sum-square. The exponent is the amplifier:
$d$ appears squared, so its error counts **twice**.

In [ ]:
u_m = 0.05e-3     # kg
u_d = 0.01e-3     # m
u_h = 0.25e-3     # m

term_m = 1 * u_m / m
term_d = 2 * u_d / d
term_h = 1 * u_h / h

print(f"m term: {term_m*100:.4f} %")
print(f"d term: {term_d*100:.4f} %   <- exponent 2 doubles the caliper's error")
print(f"h term: {term_h*100:.4f} %")

> **CHECKPOINT 2:** Report the d term as a percentage, to 4 decimals.

## 3. The budget as data: tuples + one loop

Writing three named terms was fine; nine (as in lab) would be painful. The
scalable pattern stores the whole budget as a **list of tuples** — one row
per variable: `(name, value, uncertainty, exponent)` — and loops:

In [ ]:
variables = [
    ('m', m, u_m, +1),
    ('d', d, u_d, -2),
    ('h', h, u_h, -1),
]

names    = [v[0] for v in variables]
rel      = np.array([p * u / x for name, x, u, p in variables])
contribs = np.abs(rel) * rho                 # per-variable, in kg/m^3
total    = np.sqrt(np.sum(contribs**2))

for name, cont in zip(names, contribs):
    print(f"  {name}: ±{cont:6.2f} kg/m^3")
print(f"TOTAL: ±{total:.2f} kg/m^3  ({total/rho*100:.3f} %)")

Read the loop line carefully: `for name, x, u, p in variables:` unpacks
one tuple per pass (Lab 05's unpacking, now inside a loop), and `p * u / x`
is one power-rule term. The lab's nine-variable budget for Young's modulus
is this exact cell with a longer list.

> **CHECKPOINT 3:** Report the TOTAL uncertainty in kg/m³, to 2 decimals.

## 4. Seeing the budget: `ax.bar`

A budget's job is to show *what dominates* — a bar chart does that at a
glance:

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
fig.patch.set_facecolor('white')

ax.bar(names, contribs, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Variable')
ax.set_ylabel('Contribution to $u_\\rho$ (kg/m$^3$)')
ax.grid(axis='y', linestyle='--', linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

`ax.bar(categories, heights)` is the whole API — everything else you
know about axes applies. Note what the chart says: the *ruler* measurement
dominates, even though $h$ has the smallest exponent. Instrument choice beat
exponent here — knowing which fight matters is the entire point of a budget.

> **CHECKPOINT 4:** Which variable dominates the density budget? Report its
> letter.

## 5. The bracketing trick: `v[:-1]` and `v[1:]`

In lab, every loaded reading is *bracketed* by zero-load readings, and the
readings drift slowly upward. Ten unloaded readings interleave nine loaded
ones; here is a miniature with four and three:

In [ ]:
v_unl  = np.array([2.1240, 2.1262, 2.1281, 2.1301])   # drifting zeros
v_load = np.array([2.2715, 2.3470, 2.4210])            # loaded readings

print('all but last :', v_unl[:-1])    # the zero BEFORE each load
print('all but first:', v_unl[1:])     # the zero AFTER each load

dv1 = v_load - v_unl[:-1]
dv2 = v_load - v_unl[1:]
dv  = (dv1 + dv2) / 2                  # average cancels linear drift

print('dv1   :', np.round(dv1, 4))
print('dv2   :', np.round(dv2, 4))
print('dv_avg:', np.round(dv, 4))
print('dv1 - dv2 (the drift, exposed):', np.round(dv1 - dv2, 4))

Two shifted views of one array pair every load with its neighbors — no
loop, no index bookkeeping. And notice `dv1 - dv2`: the bracketing procedure
*measures its own drift*, which the lab recycles as the uncertainty of dV.

> **CHECKPOINT 5:** Report the middle value of `dv_avg`, to 4 decimals.

## Summary: where each skill is used in lab

| Skill | Where you will use it |
|---|---|
| power rule (exponents × relative u) | the Young's-modulus budget, Part-5 |
| list-of-tuples budget + loop | `budget()` in Part-5 |
| `ax.bar` | the uncertainty-budget deliverable figure |
| `v[:-1]` / `v[1:]` pairing | the bracketed load data, Parts 3–4 |
| drift exposed by `dv1 - dv2` | your empirical u_dV, Part-5 |

See you in lab — and remember: nobody touches the wires.